In [ ]:
####################################
#ENVIRONMENT SETUP"

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr; #xr.set_options(file_cache_maxsize=1)

import sys; import os; import time; from datetime import timedelta
import pickle
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, MemoryTracker_Class

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=6)
MemoryTracker=MemoryTracker_Class()

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res, t_res):
    jobs = {
        ('1km', '5min'): 20,
        ('1km', '3min'): 30,
        ('1km', '1min'): 100,
        ('250m', '1min'): 400,
    }
    return jobs.get((res, t_res))
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#DATA IO FUNCTIONS

In [ ]:
#Getting Data
def GetData():
    if not ModelData.is_data_timestepbytimestep:
        dataNC = ModelData.OpenData()  
        dataNC = ModelData.SubsetDataVars(dataNC)
    else:
        dataNC = None
    parcelNC = ModelData.OpenParcel()
    return [dataNC,parcelNC]

#Output Directories
outputDirectory=os.path.join(mainDirectory,'Code','OUTPUT','Variable_Calculation','TimeSplitModelData')
os.makedirs(outputDirectory, exist_ok=True)

#Data Output Directories
def MakeDataDirectories(outputDirectory,res,t_res,Nz_str):
    outputDataDirectory = os.path.join(outputDirectory,
                                       f"Simulation_{ModelData.simulationNumber}_{res}_{t_res}_{Nz_str}nz",'ModelData')
    outputParcelDirectory = os.path.join(outputDirectory,
                                         f"Simulation_{ModelData.simulationNumber}_{res}_{t_res}_{Nz_str}nz",'ParcelData')
    os.makedirs(outputDataDirectory, exist_ok=True)
    os.makedirs(outputParcelDirectory, exist_ok=True)

    return outputDataDirectory, outputParcelDirectory

In [ ]:
####################################
#FUNCTIONS

In [ ]:
def WriteTimesteps(dataNC, parcelNC, loop_elements,ModelData,
                         outputDataDirectory, outputParcelDirectory):
    """
    Writes each timestep of dataNC and parcelNC to separate .h5 files,
    mirroring WriteTimestepsNetCDF but using h5py instead of xarray.to_netcdf.
    """
    
    for loop_element in tqdm(loop_elements,total=len(loop_elements),desc="Writing timesteps"):
        # Extract single timestep
        if ModelData.is_data_timestepbytimestep:
            dataT = ModelData.OpenData_SingleTime(t=loop_element).isel(time=0)
        else:
            dataT = dataNC.isel(time=loop_element)            
        parcelT = parcelNC.isel(time=loop_element) #always one big file in CM1

        # Get timeString for titles
        timeString = ModelData.timeStrings[loop_element]
        
        # Build file names (same as NetCDF version)
        outputDataFile = os.path.join(
            outputDataDirectory,
            f"cm1out_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz_{timeString}.h5"
        )
        outputParcelFile = os.path.join(
            outputParcelDirectory,
            f"cm1out_pdata_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}np_{timeString}.h5"
        )
        print(f"outputting to {outputDataFile}")

        # --- Write data timestep ---
        WriteVariables(outputDataFile, dataT)
        
        # --- Write parcel timestep ---
        WriteVariables(outputParcelFile, parcelT)

        if ModelData.is_data_timestepbytimestep:
            del dataT

def WriteVariables(outputFile, dataT):
    """
    Writes all data variables of an xarray Dataset (dataT) to a single .h5
    file (outputFile).
    """
    with h5py.File(outputFile, "w", libver="latest") as fileH5:
        buffer = None #setup buffer
        for varName, da in tqdm(dataT.data_vars.items(), miniters=5, desc="Writing variables"):
            varData = da.values  # read this variable from disk

            #Copying varData to fixed memory location (mainly for saving memory for next timestep)
            if buffer is None or buffer.shape != varData.shape: 
                buffer = np.empty(varData.shape, dtype="float32")  # allocated buffer to memory
            np.copyto(buffer, varData) #copy varData to buffer
            del varData  # release raw data immediately

            #Saving
            dset = fileH5.create_dataset(varName, data=buffer, compression=None)
            for attr, val in da.attrs.items():
                dset.attrs[attr] = val

            MemoryTracker.GetMemGB()

In [ ]:
####################################
#RUNNING

In [ ]:
#getting data
[dataNC,parcelNC] = GetData()

#getting output directories
[outputDataDirectory, outputParcelDirectory] = MakeDataDirectories(outputDirectory,ModelData.res,ModelData.t_res,ModelData.Nz_str)

#running
WriteTimesteps(dataNC, parcelNC, loop_elements,ModelData, outputDataDirectory,outputParcelDirectory)